# SiteX: CSV to SpatiaLite Migration Pipeline

This notebook manages the transition from independent, unindexed CSV files to a unified, spatially-indexed SQLite database (`sitex_geospatial.db`). This improves data extraction speed exponentially and removes the O(n*m) complexity in feature engineering.

### Requirements:
Ensure you have the required packages installed in your `.venv`:
```bash
pip install pandas spatialite
```

In [11]:
import sqlite3
import pandas as pd
import os
from pathlib import Path

# Paths configuration relative to backend directory
WORKSPACE_DIR = Path('..')
DB_PATH = WORKSPACE_DIR / 'sitex_geospatial.db'
CSV_DIR = WORKSPACE_DIR / 'DataEngineering' / 'CSV_Reference'

# Mapping CSV files to POI type
FILE_MAPPINGS = {
    'banks.csv': 'bank',
    'education.csv': 'school',
    'health.csv': 'hospital',
    'temples.csv': 'temple',
    'other.csv': 'other'
}

print(f"Database will be created at: {DB_PATH.resolve()}")
print(f"Looking for CSVs in: {CSV_DIR.resolve()}")

Database will be created at: /mnt/windows/Users/taman/projects/SiteX/backend/sitex_geospatial.db
Looking for CSVs in: /mnt/windows/Users/taman/projects/SiteX/backend/DataEngineering/CSV_Reference


## 1. Database Initialization
Create an empty database and enable the SpatiaLite extension which provides functions like `ST_Distance` and R-tree clustering indexing.

In [12]:
# Start fresh
if DB_PATH.exists():
    try:
        os.remove(DB_PATH)
        print("Removed exiting database for a clean start.")
    except PermissionError:
        print("Database is currently locked. Ensure no other applications are using it.")

# Establish connection
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Enable spatial capabilities
conn.enable_load_extension(True)
try:
    conn.load_extension('mod_spatialite') 
    print("✓ SpatiaLite extension enabled.")
except Exception as e:
    print(f"⚠️ Error loading SpatiaLite: {e}")
conn.enable_load_extension(False)

# Initialize standard spatial metadata
cursor.execute("SELECT InitSpatialMetaData(1)")
conn.commit()
print("✓ Spatial metadata initialized.")

Removed exiting database for a clean start.
✓ SpatiaLite extension enabled.
✓ Spatial metadata initialized.


## 2. Defining Schema with Spatial Geometry Columns
We will define the core architecture here. For spatial speed, all geometries are wrapped up with `CreateSpatialIndex()` functions which internally manage an R-tree logic layout.

In [13]:
# Drop tables if exist
cursor.execute("DROP TABLE IF EXISTS cafes")
cursor.execute("DROP TABLE IF EXISTS pois")

print("Building tables...")

# CAFE TABLE
cursor.execute("""
    CREATE TABLE cafes (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT,
        lat REAL NOT NULL,
        lon REAL NOT NULL,
        revenue REAL,
        opening_year INTEGER,
        description TEXT
    )
""")
# Add Geometry and Index (EPSG 4326 = standard GPS coordinates format)
cursor.execute("SELECT AddGeometryColumn('cafes', 'geometry', 4326, 'POINT', 'XY')")
cursor.execute("SELECT CreateSpatialIndex('cafes', 'geometry')")


# GENERIC POI TABLE (Banks, Education, Health...)
cursor.execute("""
    CREATE TABLE pois (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        poi_type TEXT NOT NULL,
        name TEXT,
        lat REAL NOT NULL,
        lon REAL NOT NULL,
        subtype TEXT,
        address TEXT
    )
""")
cursor.execute("SELECT AddGeometryColumn('pois', 'geometry', 4326, 'POINT', 'XY')")
cursor.execute("SELECT CreateSpatialIndex('pois', 'geometry')")


# CAFE METRICS (Pre-computed edge connections for GNN mapping)
cursor.execute("""
    CREATE TABLE cafe_metrics (
        cafe_id INTEGER PRIMARY KEY,
        nearest_bank_dist REAL,
        nearest_school_dist REAL,
        poi_count_500m INTEGER,
        road_network_node_id INTEGER,
        FOREIGN KEY(cafe_id) REFERENCES cafes(id)
    )
""")

conn.commit()
print("✓ Schema definitions successful and geometry columns created!")

Building tables...
✓ Schema definitions successful and geometry columns created!


## 3. Extract & Load Process
Iterate through the existing target CSVs and inject them into their destination schemas, generating a `POINT()` mapping via SQLite’s geometries.

In [14]:
def map_csv_to_table(filename, table_name, poi_type=None):
    csv_file = CSV_DIR / filename
    if not csv_file.exists():
        print(f"Skipping {filename}: File not found.")
        return 0
        
    df = pd.read_csv(csv_file)
    count = 0
    
    for _, row in df.iterrows():
        lat = float(row.get('lat', 0))
        lon = float(row.get('lon', 0))
        name = row.get('name', 'Unknown')
        
        if table_name == 'cafes':
            rev = float(row.get('revenue', 0)) if pd.notna(row.get('revenue')) else None
            year = int(row.get('opening_year', 0)) if pd.notna(row.get('opening_year')) else None
            desc = row.get('description', '')
            
            cursor.execute("""
                INSERT INTO cafes (name, lat, lon, revenue, opening_year, description) 
                VALUES (?, ?, ?, ?, ?, ?)
            """, (name, lat, lon, rev, year, desc))
            
            last_id = cursor.lastrowid
            # Populate Spatial Mapping Geometry
            cursor.execute(f"UPDATE cafes SET geometry = GeomFromText('POINT({lon} {lat})', 4326) WHERE id = {last_id}")
            
        else:
            subtype = row.get('category', '')
            address = row.get('address', '')
            
            cursor.execute("""
                INSERT INTO pois (poi_type, name, lat, lon, subtype, address) 
                VALUES (?, ?, ?, ?, ?, ?)
            """, (poi_type, name, lat, lon, subtype, address))
            
            last_id = cursor.lastrowid
            # Populate Spatial Mapping Geometry
            cursor.execute(f"UPDATE pois SET geometry = GeomFromText('POINT({lon} {lat})', 4326) WHERE id = {last_id}")
            
        count += 1
        
    conn.commit()
    print(f"✓ Loaded {count} rows from {filename} -> `{table_name}`")
    return count

print("Migration in progress...")
# Migrate Cafes
map_csv_to_table('cafes.csv', 'cafes')

# Migrate POIs Contexts
for file_name, poi_flag in FILE_MAPPINGS.items():
    map_csv_to_table(file_name, 'pois', poi_type=poi_flag)
print("Migration completed.")

Migration in progress...
✓ Loaded 4052 rows from cafes.csv -> `cafes`
✓ Loaded 613 rows from banks.csv -> `pois`
✓ Loaded 1804 rows from education.csv -> `pois`
✓ Loaded 1575 rows from health.csv -> `pois`
✓ Loaded 2200 rows from temples.csv -> `pois`
✓ Loaded 1913 rows from other.csv -> `pois`
Migration completed.


## 4. Verification Check and Real-world SQL test
Execute a 500m radius check natively via SpatiaLite around the first mapped cafe to confirm R-Tree functionality.

In [15]:
print("--- DATABASE STATISTICS ---\n")
cursor.execute("SELECT COUNT(*) FROM cafes")
print(f"Total Cafes Loaded: {cursor.fetchone()[0]}")

cursor.execute("SELECT poi_type, COUNT(*) FROM pois GROUP BY poi_type")
print("\nPOI Distributions:")
for row in cursor.fetchall():
    print(f"  {row[0].capitalize()}: {row[1]}")

print("\n\n--- TESTING SPATIAL INDEX PERFORMANCE ---")

# Let's see the performance vs looping!
cursor.execute("SELECT id, name FROM cafes LIMIT 1")
cafe = cursor.fetchone()

if cafe:
    print(f"\nFinding POI density within 500m of Cafe #{cafe[0]} '{cafe[1]}':")
    
    # Notice how we let SQL perform the radius search directly using geometry points!
    cursor.execute("""
        SELECT poi_type, COUNT(*) as count 
        FROM pois 
        WHERE ST_Distance(geometry, (SELECT geometry FROM cafes WHERE id = ?)) < 500
        GROUP BY poi_type
    """, (cafe[0],))
    
    for row in cursor.fetchall():
        print(f"  Found {row[1]} {row[0].capitalize()}s within 500m.")
else:
    print("No cafes found to run test with.")
    
# Clean Exit
conn.close()

--- DATABASE STATISTICS ---

Total Cafes Loaded: 4052

POI Distributions:
  Bank: 613
  Hospital: 1575
  Other: 1913
  School: 1804
  Temple: 2200


--- TESTING SPATIAL INDEX PERFORMANCE ---

Finding POI density within 500m of Cafe #1 'Happy Hills Adventure':
  Found 613 Banks within 500m.
  Found 1575 Hospitals within 500m.
  Found 1913 Others within 500m.
  Found 1804 Schools within 500m.
  Found 2200 Temples within 500m.
